# Lab # 8: Convolutional Neural Network (CNN) on MNIST Dataset

In [3]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras import models, layers, optimizers
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

NameError: name '_ndimage_api' is not defined

## Task 1: Load and Explore the MNIST Dataset

In [ ]:
# 1. Load MNIST Dataset
(train_images, train_labels), (test_images, test_labels) = mnist.load_data()
print('Train images:', train_images.shape, ' Test images:', test_images.shape)

# 2. Visualize sample images and their labels
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(train_images[i], cmap='gray')
    plt.title('Label: ' + str(train_labels[i]))
    plt.axis('off')
plt.tight_layout()
plt.show()

# 3. Reshape to (N, 28, 28, 1) for the CNN and normalize pixel values to [0, 1]
train_images = train_images.reshape((60000, 28, 28, 1)).astype('float32') / 255
test_images = test_images.reshape((10000, 28, 28, 1)).astype('float32') / 255

# 4. One-Hot Encode Class Labels
train_labels_by_cat = to_categorical(train_labels)
test_labels_by_cat = to_categorical(test_labels)

## Task 2: Build the CNN Model

In [ ]:
def build_cnn(filters=32, lr=0.001):
    model = models.Sequential()
    model.add(layers.Input(shape=(28, 28, 1)))
    model.add(layers.Conv2D(filters, (3, 3), activation='relu'))        # convolutional layer 1
    model.add(layers.MaxPooling2D((2, 2)))                              # pooling layer 1
    model.add(layers.Conv2D(filters * 2, (3, 3), activation='relu'))    # convolutional layer 2
    model.add(layers.MaxPooling2D((2, 2)))                              # pooling layer 2
    model.add(layers.Flatten())
    model.add(layers.Dense(128, activation='relu'))                     # fully connected layer
    model.add(layers.Dense(10, activation='softmax'))                   # output layer
    model.compile(optimizer=optimizers.Adam(learning_rate=lr),
                  loss='categorical_crossentropy', metrics=['accuracy'])
    return model

model = build_cnn()
model.summary()

## Task 3: Compile and Train the Model

In [ ]:
# Train with 10% of the training data held out as the validation set
history = model.fit(train_images, train_labels_by_cat,
                    epochs=5, batch_size=128, validation_split=0.1)

In [ ]:
# Plot training and validation accuracy and loss over epochs
plt.figure(figsize=(11, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Training accuracy')
plt.plot(history.history['val_accuracy'], label='Validation accuracy')
plt.xlabel('Epoch'); plt.ylabel('Accuracy'); plt.title('Accuracy'); plt.legend()
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Training loss')
plt.plot(history.history['val_loss'], label='Validation loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss'); plt.legend()
plt.tight_layout()
plt.show()

## Task 4: Evaluate the Model

In [ ]:
# 1. Evaluate on the test set
test_loss, test_acc = model.evaluate(test_images, test_labels_by_cat)
print('Test loss:', test_loss, 'Test accuracy:', test_acc)

# 2. Predict classes and generate the classification report
pred = np.argmax(model.predict(test_images), axis=1)
print(classification_report(test_labels, pred, digits=4))

# 3. Confusion matrix
cm = confusion_matrix(test_labels, pred)
ConfusionMatrixDisplay(cm, display_labels=range(10)).plot(cmap='Blues')
plt.title('Confusion Matrix (MNIST test set)')
plt.show()

## Task 5: Hyperparameter Tuning

In [ ]:
# One hyperparameter is changed at a time; the others stay at the base values
base = dict(filters=32, lr=0.001, batch=128, epochs=3)

def run(**change):
    p = {**base, **change}
    m = build_cnn(p['filters'], p['lr'])
    m.fit(train_images, train_labels_by_cat, epochs=p['epochs'],
          batch_size=p['batch'], verbose=0)
    loss, acc = m.evaluate(test_images, test_labels_by_cat, verbose=0)
    return loss, acc * 100

experiments = (
    [('Learning rate', v, {'lr': v}) for v in [0.01, 0.001, 0.0001]] +
    [('Batch size', v, {'batch': v}) for v in [32, 128, 512]] +
    [('Epochs', v, {'epochs': v}) for v in [1, 3, 5]] +
    [('Filters', v, {'filters': v}) for v in [8, 32, 64]]
)

print(f"{'Hyperparameter':15s} {'Value':>8s} {'Test loss':>10s} {'Accuracy':>9s}")
for name, value, change in experiments:
    loss, acc = run(**change)
    print(f'{name:15s} {str(value):>8s} {loss:10.4f} {acc:8.2f}%')